# Week 2: Linear Models (Lasso, Ridge, Elastic Net) on Box Office Data

This notebook explores linear regression, Lasso, Ridge, and Elastic Net models on the `box_office` dataset. The target variable is **Rating**.

## 1. Imports and Data Loading

In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.linear_model import LinearRegression, Lasso, Ridge, ElasticNet
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.preprocessing import StandardScaler

# Load data
box_office = pd.read_csv('Box_office_clean.csv')
box_office.head()

,Rank,Release Group,$Worldwide,$Domestic,Domestic %,$Foreign,Foreign %,Year,Genres,Rating,Vote_Count,Original_Language,Production_Countries,Original_Language_encoded
0,1,Mission: Impossible II,546388108.0,215409889.0,39.4,330978219.0,60.6,2000,"Adventure, Action, Thriller",6.126,6741.0,en,United States of America,6.0
1,2,Gladiator,460583960.0,187705427.0,40.8,272878533.0,59.2,2000,"Action, Drama, Adventure",8.217,19032.0,en,"United Kingdom, United States of America",6.0
2,3,Cast Away,429632142.0,233632142.0,54.4,196000000.0,45.6,2000,"Adventure, Drama",7.663,11403.0,en,United States of America,6.0
3,4,What Women Want,374111707.0,182811707.0,48.9,191300000.0,51.1,2000,"Comedy, Romance",6.450,3944.0,en,"United Kingdom, United States of America",6.0
4,5,Dinosaur,349822765.0,137748063.0,39.4,212074702.0,60.6,2000,"Animation, Family, Adventure",6.544,2530.0,en,United States of America,6.0


## 2. Data Preprocessing
- Inspect data
- Handle missing values
- Encode categorical variables
- Feature selection/engineering

Data already cleaned

In [3]:
box_office.info()
box_office.isnull().sum()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4797 entries, 0 to 4796
Data columns (total 14 columns):
 #   Column                     Non-Null Count  Dtype  
---  ------                     --------------  -----  
 0   Rank                       4797 non-null   int64  
 1   Release Group              4797 non-null   object 
 2   $Worldwide                 4797 non-null   float64
 3   $Domestic                  4797 non-null   float64
 4   Domestic %                 4797 non-null   float64
 5   $Foreign                   4797 non-null   float64
 6   Foreign %                  4797 non-null   float64
 7   Year                       4797 non-null   int64  
 8   Genres                     4797 non-null   object 
 9   Rating                     4797 non-null   float64
 10  Vote_Count                 4797 non-null   float64
 11  Original_Language          4797 non-null   object 
 12  Production_Countries       4797 non-null   object 
 13  Original_Language_encoded  4797 non-null   float

Rank                         0
Release Group                0
$Worldwide                   0
$Domestic                    0
Domestic %                   0
$Foreign                     0
Foreign %                    0
Year                         0
Genres                       0
Rating                       0
Vote_Count                   0
Original_Language            0
Production_Countries         0
Original_Language_encoded    0
dtype: int64

## 3. Train-Test Split & Scaling

In [6]:
categorical_cols = box_office.select_dtypes(include=['object']).columns.tolist()
categorical_cols = [col for col in categorical_cols if col != 'Rating']
box_office_encoded = pd.get_dummies(box_office, columns=categorical_cols, drop_first=True)

X = box_office_encoded.drop('Rating', axis=1)
y = box_office_encoded['Rating']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Scale features
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

## 4. Linear Regression Baseline

In [7]:
lr = LinearRegression()
lr.fit(X_train_scaled, y_train)
y_pred_lr = lr.predict(X_test_scaled)
print("Linear Regression RMSE:", np.sqrt(mean_squared_error(y_test, y_pred_lr)))
print("Linear Regression R^2:", r2_score(y_test, y_pred_lr))

Linear Regression RMSE: 0.8857720273141407
Linear Regression R^2: 0.12205635886738808


## 5. Lasso Regression

In [8]:
# Tune alpha with GridSearchCV
lasso = Lasso(max_iter=10000)
params = {'alpha': np.logspace(-3, 1, 20)}
lasso_cv = GridSearchCV(lasso, params, cv=5)
lasso_cv.fit(X_train_scaled, y_train)
print("Best alpha for Lasso:", lasso_cv.best_params_['alpha'])
y_pred_lasso = lasso_cv.predict(X_test_scaled)
print("Lasso RMSE:", np.sqrt(mean_squared_error(y_test, y_pred_lasso)))
print("Lasso R^2:", r2_score(y_test, y_pred_lasso))

Best alpha for Lasso: 0.004281332398719396
Lasso RMSE: 0.8337105354276702
Lasso R^2: 0.2222262266511016


## 6. Ridge Regression

In [9]:
ridge = Ridge(max_iter=10000)
params = {'alpha': np.logspace(-3, 1, 20)}
ridge_cv = GridSearchCV(ridge, params, cv=5)
ridge_cv.fit(X_train_scaled, y_train)
print("Best alpha for Ridge:", ridge_cv.best_params_['alpha'])
y_pred_ridge = ridge_cv.predict(X_test_scaled)
print("Ridge RMSE:", np.sqrt(mean_squared_error(y_test, y_pred_ridge)))
print("Ridge R^2:", r2_score(y_test, y_pred_ridge))

Best alpha for Ridge: 10.0
Ridge RMSE: 0.8724170813148304
Ridge R^2: 0.1483306154701436


## 7. Elastic Net Regression

In [10]:
elastic = ElasticNet(max_iter=10000)
params = {
    'alpha': np.logspace(-3, 1, 10),
    'l1_ratio': np.linspace(0.1, 0.9, 9)
}
elastic_cv = GridSearchCV(elastic, params, cv=5)
elastic_cv.fit(X_train_scaled, y_train)
print("Best params for Elastic Net:", elastic_cv.best_params_)
y_pred_elastic = elastic_cv.predict(X_test_scaled)
print("Elastic Net RMSE:", np.sqrt(mean_squared_error(y_test, y_pred_elastic)))
print("Elastic Net R^2:", r2_score(y_test, y_pred_elastic))

Best params for Elastic Net: {'alpha': np.float64(0.021544346900318832), 'l1_ratio': np.float64(0.1)}
Elastic Net RMSE: 0.8409288852454814
Elastic Net R^2: 0.20869983523084445


## Conclusions

Lasso performed best